In [1]:
def romanized_text_pipeline(input_file, output_file=None, english_threshold=0.4, min_genuine_words=4, save_original_case=False, show_progress=True):
    """
    Complete pipeline to extract romanized sentences from any text file
    
    Parameters:
    - input_file: Path to input text file
    - output_file: Path to save cleaned sentences (optional)
    - english_threshold: Threshold for English words (default 0.4 = 40%)
    - min_genuine_words: Minimum number of genuine (non-English) words required (default 4)
    - save_original_case: Whether to save in original case or lowercase
    - show_progress: Show progress bar for large files (default True)
    
    Returns:
    - romanized_sentences: List of cleaned romanized sentences
    - statistics: Dictionary with processing statistics
    """
    import re
    import string
    import nltk
    from nltk.corpus import words
    from tqdm import tqdm  # For progress bars
    
    # Ensure NLTK words corpus is available
    try:
        english_words = set(words.words())
    except LookupError:
        print("Downloading NLTK words corpus...")
        nltk.download('words')
        english_words = set(words.words())
    
    print(f"🔄 Processing file: {input_file}")
    print(f"📊 English threshold: {english_threshold*100:.0f}%")
    print(f"📝 Minimum genuine words: {min_genuine_words}")
    print("-" * 50)
    
    # Step 1: Load sentences
    sentences = []
    try:
        with open(input_file, 'r', encoding='utf-8') as file:
            for line in file:
                sentence = line.strip()
                if sentence:
                    sentences.append(sentence)
        print(f"✅ Loaded {len(sentences)} sentences from {input_file}")
    except FileNotFoundError:
        print(f"❌ Error: File {input_file} not found")
        return [], {}
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        return [], {}
    
    # Step 2: Text cleaning functions
    def remove_emojis(text):
        emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"  # emoticons
            "\U0001F300-\U0001F5FF"  # symbols & pictographs
            "\U0001F680-\U0001F6FF"  # transport & map symbols
            "\U0001F1E0-\U0001F1FF"  # flags (iOS)
            "\U00002500-\U00002BEF"  # chinese characters
            "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251"
            "\U0001f926-\U0001f937"
            "\U00010000-\U0010ffff"
            "\u2640-\u2642"
            "\u2600-\u2B55"
            "\u200d\u23cf\u23e9\u231a\ufe0f\u3030"
            "]+", flags=re.UNICODE)
        return re.sub(r'\s+', ' ', emoji_pattern.sub(r'', text)).strip()
    
    def clean_text_separators(text):
        # Separate words joined by periods or parentheses
        text = re.sub(r'([a-zA-Z])\.([a-zA-Z])', r'\1 \2', text)
        text = re.sub(r'([a-zA-Z])\(([a-zA-Z])', r'\1 \2', text)
        text = re.sub(r'([a-zA-Z])\)([a-zA-Z])', r'\1 \2', text)
        text = re.sub(r'[()]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()
    
    # Step 3: Apply text cleaning
    sentences_clean = []
    emoji_count = 0
    separator_count = 0
    
    # Use progress bar for text cleaning if enabled and large dataset
    if show_progress and len(sentences) > 1000:
        sentence_iterator = tqdm(sentences, desc="🧹 Cleaning text", unit="sentences")
    else:
        sentence_iterator = sentences
    
    for sentence in sentence_iterator:
        # Remove emojis
        clean_sentence = remove_emojis(sentence)
        if clean_sentence != sentence:
            emoji_count += 1
        
        # Clean separators
        original_clean = clean_sentence
        clean_sentence = clean_text_separators(clean_sentence)
        if clean_sentence != original_clean:
            separator_count += 1
        
        if clean_sentence.strip():
            sentences_clean.append(clean_sentence)
    
    print(f"🧹 Text cleaning completed:")
    print(f"   - Emojis removed: {emoji_count} sentences")
    print(f"   - Separators fixed: {separator_count} sentences")
    print(f"   - Clean sentences: {len(sentences_clean)}")
    
    # Step 4: Calculate English ratios
    def calculate_english_ratio(sentence):
        translator = str.maketrans('', '', string.punctuation)
        cleaned_sentence = sentence.translate(translator)
        words = cleaned_sentence.lower().split()
        words = [word for word in words if word.strip() and not word.isdigit()]
        
        if not words:
            return 0.0, 0, 0, 0  # ratio, english_count, total_count, genuine_count
        
        english_word_count = sum(1 for word in words if word in english_words)
        total_word_count = len(words)
        genuine_word_count = total_word_count - english_word_count  # Non-English words
        
        ratio = english_word_count / total_word_count if total_word_count > 0 else 0.0
        
        return ratio, english_word_count, total_word_count, genuine_word_count
    
    sentence_analysis = []
    
    # Use progress bar for English ratio calculation if enabled and large dataset
    if show_progress and len(sentences_clean) > 1000:
        analysis_iterator = tqdm(enumerate(sentences_clean), 
                               desc="📊 Analyzing English ratios", 
                               total=len(sentences_clean), 
                               unit="sentences")
    else:
        analysis_iterator = enumerate(sentences_clean)
    
    for i, sentence in analysis_iterator:
        ratio, eng_count, total_count, genuine_count = calculate_english_ratio(sentence)
        sentence_analysis.append({
            'index': i,
            'original_sentence': sentence,
            'english_ratio': ratio,
            'english_words': eng_count,
            'total_words': total_count,
            'genuine_words': genuine_count
        })
    
    # Step 5: Filter romanized sentences
    print("🔍 Filtering romanized sentences...")
    
    # Filter by English ratio AND minimum genuine words
    romanized_sentences = [analysis for analysis in sentence_analysis 
                          if analysis['english_ratio'] < english_threshold 
                          and analysis['genuine_words'] >= min_genuine_words]
    
    # Count sentences removed for different reasons
    high_english = len([a for a in sentence_analysis if a['english_ratio'] >= english_threshold])
    low_genuine = len([a for a in sentence_analysis if a['genuine_words'] < min_genuine_words])
    both_issues = len([a for a in sentence_analysis 
                      if a['english_ratio'] >= english_threshold and a['genuine_words'] < min_genuine_words])
    
    removed_count = len(sentences_clean) - len(romanized_sentences)
    
    print(f"🔍 Filtering analysis:")
    print(f"   - Threshold: <{english_threshold*100:.0f}% English AND ≥{min_genuine_words} genuine words")
    print(f"   - Removed for high English (≥{english_threshold*100:.0f}%): {high_english}")
    print(f"   - Removed for low genuine words (<{min_genuine_words}): {low_genuine}")
    print(f"   - Removed for both issues: {both_issues}")
    print(f"   - Total sentences removed: {removed_count}")
    print(f"   - Romanized sentences kept: {len(romanized_sentences)}")
    print(f"   - Success rate: {len(romanized_sentences)/len(sentences_clean)*100:.1f}%")
    
    # Step 6: Extract final text
    def process_to_lowercase(sentence):
        translator = str.maketrans('', '', string.punctuation)
        cleaned_sentence = sentence.translate(translator)
        return cleaned_sentence.lower().strip()
    
    if save_original_case:
        final_sentences = [analysis['original_sentence'] for analysis in romanized_sentences]
    else:
        final_sentences = [process_to_lowercase(analysis['original_sentence']) 
                          for analysis in romanized_sentences]
    
    # Step 7: Save to file if specified
    if output_file:
        try:
            print(f"💾 Saving to {output_file}...")
            with open(output_file, 'w', encoding='utf-8') as file:
                # Use progress bar for large files when saving
                if show_progress and len(final_sentences) > 10000:
                    save_iterator = tqdm(final_sentences, desc="💾 Saving file", unit="sentences")
                else:
                    save_iterator = final_sentences
                
                for sentence in save_iterator:
                    file.write(sentence + '\n')
            print(f"✅ Saved {len(final_sentences)} sentences to {output_file}")
        except Exception as e:
            print(f"❌ Error saving file: {e}")
    
    # Step 8: Generate statistics
    kept_ratios = [analysis['english_ratio'] for analysis in romanized_sentences]
    kept_genuine = [analysis['genuine_words'] for analysis in romanized_sentences]
    
    statistics = {
        'input_file': input_file,
        'output_file': output_file,
        'original_sentences': len(sentences),
        'cleaned_sentences': len(sentences_clean),
        'romanized_sentences': len(romanized_sentences),
        'removed_sentences': removed_count,
        'removed_high_english': high_english,
        'removed_low_genuine': low_genuine,
        'removed_both_issues': both_issues,
        'success_rate': len(romanized_sentences)/len(sentences_clean)*100,
        'avg_english_ratio': sum(kept_ratios)/len(kept_ratios)*100 if kept_ratios else 0,
        'max_english_ratio': max(kept_ratios)*100 if kept_ratios else 0,
        'min_english_ratio': min(kept_ratios)*100 if kept_ratios else 0,
        'avg_genuine_words': sum(kept_genuine)/len(kept_genuine) if kept_genuine else 0,
        'min_genuine_words': min(kept_genuine) if kept_genuine else 0,
        'max_genuine_words': max(kept_genuine) if kept_genuine else 0,
        'emoji_fixes': emoji_count,
        'separator_fixes': separator_count
    }
    
    print(f"\n📈 Final Statistics:")
    print(f"   - Average English ratio: {statistics['avg_english_ratio']:.1f}%")
    print(f"   - English ratio range: {statistics['min_english_ratio']:.1f}% - {statistics['max_english_ratio']:.1f}%")
    print(f"   - Average genuine words: {statistics['avg_genuine_words']:.1f}")
    print(f"   - Genuine words range: {statistics['min_genuine_words']:.0f} - {statistics['max_genuine_words']:.0f}")
    print("✅ Pipeline completed successfully!")
    
    return final_sentences, statistics

# Example usage:
print("📋 Pipeline Function Created!")
print("\n🔧 Usage Examples:")
print("# Basic usage:")
print("sentences, stats = romanized_text_pipeline('input.txt')")
print("\n# With output file:")
print("sentences, stats = romanized_text_pipeline('input.txt', 'output_romanized.txt')")
print("\n# Custom threshold (30%):")
print("sentences, stats = romanized_text_pipeline('input.txt', english_threshold=0.3)")
print("\n# Keep original case:")
print("sentences, stats = romanized_text_pipeline('input.txt', save_original_case=True)")

📋 Pipeline Function Created!

🔧 Usage Examples:
# Basic usage:
sentences, stats = romanized_text_pipeline('input.txt')

# With output file:
sentences, stats = romanized_text_pipeline('input.txt', 'output_romanized.txt')

# Custom threshold (30%):
sentences, stats = romanized_text_pipeline('input.txt', english_threshold=0.3)

# Keep original case:
sentences, stats = romanized_text_pipeline('input.txt', save_original_case=True)


In [ ]:
# 🛠️ ADDITIONAL UTILITY FUNCTIONS

def batch_process_files(file_list, output_dir='processed', english_threshold=0.4, show_progress=True):
    """
    Process multiple files at once with progress tracking
    """
    import os
    from tqdm import tqdm
    
    os.makedirs(output_dir, exist_ok=True)
    
    results = {}
    
    # Progress bar for batch processing
    if show_progress:
        file_iterator = tqdm(file_list, desc="📁 Processing files", unit="files")
    else:
        file_iterator = file_list
    
    for input_file in file_iterator:
        filename = os.path.basename(input_file)
        name, ext = os.path.splitext(filename)
        output_file = os.path.join(output_dir, f"{name}_romanized{ext}")
        
        if not show_progress:
            print(f"
🔄 Processing: {input_file}")
        
        sentences, stats = romanized_text_pipeline(
            input_file, output_file, english_threshold, show_progress=show_progress
        )
        results[input_file] = {'sentences': len(sentences), 'stats': stats}
    
    return results

def quick_romanize(input_file, threshold=0.4, show_progress=True):
    """
    Quick one-liner to romanize any file with optional progress tracking
    """
    sentences, _ = romanized_text_pipeline(
        input_file, english_threshold=threshold, show_progress=show_progress
    )
    return sentences

def analyze_text_quality(sentences, sample_size=50):
    """
    Analyze the quality of extracted sentences
    """
    import random
    
    if len(sentences) < sample_size:
        sample = sentences
    else:
        sample = random.sample(sentences, sample_size)
    
    print(f"📊 QUALITY ANALYSIS (Sample: {len(sample)} sentences)")
    print("-" * 50)
    
    # Length analysis
    lengths = [len(sentence.split()) for sentence in sample]
    print(f"📏 Word count - Avg: {sum(lengths)/len(lengths):.1f}, "
          f"Min: {min(lengths)}, Max: {max(lengths)}")
    
    # Character analysis
    char_lengths = [len(sentence) for sentence in sample]
    print(f"📝 Character count - Avg: {sum(char_lengths)/len(char_lengths):.1f}, "
          f"Min: {min(char_lengths)}, Max: {max(char_lengths)}")
    
    # Sample sentences by length
    sample.sort(key=len)
    print(f"\n📋 Sample Sentences:")
    print(f"Shortest: {sample[0]}")
    print(f"Medium:   {sample[len(sample)//2]}")
    print(f"Longest:  {sample[-1]}")

def quick_romanize(input_file, threshold=0.4):
    """
    Quick one-liner to romanize any file
    """
    sentences, _ = romanized_text_pipeline(input_file, english_threshold=threshold)
    return sentences

# 🎯 USAGE EXAMPLES
print("🛠️ Additional utility functions created!")
print("\n📚 Quick Reference:")
print("• batch_process_files(['file1.txt', 'file2.txt'])  # Process multiple files")
print("• analyze_text_quality(sentences)                  # Analyze sentence quality") 
print("• quick_romanize('file.txt', 0.3)                  # Quick processing")
print("\n✨ Ready to process any text file into romanized sentences!")

🛠️ Additional utility functions created!

📚 Quick Reference:
• batch_process_files(['file1.txt', 'file2.txt'])  # Process multiple files
• analyze_text_quality(sentences)                  # Analyze sentence quality
• quick_romanize('file.txt', 0.3)                  # Quick processing

✨ Ready to process any text file into romanized sentences!


In [3]:
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.0 MB/s eta 0:00:004.3 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 kB 5.8 MB/s eta 0:00:00 MB/s eta 0:00:01:01


In [2]:
# 📊 INSTALL PROGRESS BAR SUPPORT
try:
    import tqdm
    print("✅ tqdm already installed - progress bars ready!")
except ImportError:
    print("📦 Installing tqdm for progress bars...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    print("✅ tqdm installed successfully!")

# 🚀 ENHANCED PIPELINE DEMO WITH PROGRESS BARS
print("\n" + "="*60)
print("🎯 ENHANCED PIPELINE WITH PROGRESS BARS")
print("="*60)

# Test with progress bars enabled
enhanced_sentences, enhanced_stats = romanized_text_pipeline(
    input_file='TechLekhYT_comments.txt',
    output_file='TechLekhYT_comments_romanized.txt',
    english_threshold=0.4,
    save_original_case=False,
    show_progress=True  # Enable progress bars
)

print(f"\n🎊 ENHANCED PROCESSING COMPLETE!")
print(f"📊 Progress bars shown for: Text cleaning, English analysis")
print(f"💾 File saved with progress tracking: romanized_enhanced.txt")

# 📋 PROGRESS BAR FEATURES
print(f"\n📋 PROGRESS BAR FEATURES:")
print("• 🧹 Text cleaning progress (emoji removal, separator fixes)")
print("• 📊 English ratio analysis progress") 
print("• 💾 File saving progress (for large outputs >10k sentences)")
print("• ⚡ Auto-enabled for files with >1000 sentences")
print("• 🎛️ Can be disabled with show_progress=False")

# 🔧 PERFORMANCE TIPS FOR LARGE FILES
print(f"\n🔧 LARGE FILE PERFORMANCE TIPS:")
print("• Files >100k sentences: Progress bars auto-enabled")
print("• Memory efficient: Processes line by line")
print("• Estimated time: ~1000 sentences per second")
print("• For 1M sentences: ~15-20 minutes expected")

print(f"\n✨ Ready for large-scale processing with visual feedback!")

✅ tqdm already installed - progress bars ready!

🎯 ENHANCED PIPELINE WITH PROGRESS BARS
🔄 Processing file: TechLekhYT_comments.txt
📊 English threshold: 40%
📝 Minimum genuine words: 4
--------------------------------------------------
✅ Loaded 13333 sentences from TechLekhYT_comments.txt


🧹 Cleaning text: 100%|██████████| 13333/13333 [00:00<00:00, 57775.70sentences/s]


🧹 Text cleaning completed:
   - Emojis removed: 3054 sentences
   - Separators fixed: 679 sentences
   - Clean sentences: 13147


📊 Analyzing English ratios: 100%|██████████| 13147/13147 [00:00<00:00, 68949.26sentences/s]

🔍 Filtering romanized sentences...
🔍 Filtering analysis:
   - Threshold: <40% English AND ≥4 genuine words
   - Removed for high English (≥40%): 9841
   - Removed for low genuine words (<4): 7895
   - Removed for both issues: 6867
   - Total sentences removed: 10869
   - Romanized sentences kept: 2278
   - Success rate: 17.3%
💾 Saving to TechLekhYT_comments_romanized.txt...
✅ Saved 2278 sentences to TechLekhYT_comments_romanized.txt

📈 Final Statistics:
   - Average English ratio: 25.0%
   - English ratio range: 0.0% - 39.5%
   - Average genuine words: 7.2
   - Genuine words range: 4 - 38
✅ Pipeline completed successfully!

🎊 ENHANCED PROCESSING COMPLETE!
📊 Progress bars shown for: Text cleaning, English analysis
💾 File saved with progress tracking: romanized_enhanced.txt

📋 PROGRESS BAR FEATURES:
• 🧹 Text cleaning progress (emoji removal, separator fixes)
• 📊 English ratio analysis progress
• 💾 File saving progress (for large outputs >10k sentences)
• ⚡ Auto-enabled for files with >10

## mix highlights nepal 1 and 2

In [6]:
import random

# Input files
file1 = "youtube_comments_highlightsNepal_romanized.txt"
file2 = "youtube_comments_highlightsNepal2_romanized.txt"

# Output file
output_file = "merged_comments_highlightsNepal.txt"

# Read all comments
comments = []
for fname in [file1, file2]:
    with open(fname, "r", encoding="utf-8") as f:
        comments.extend(f.readlines())

# Remove duplicates and strip whitespace
comments = [c.strip() for c in comments if c.strip()]
comments = list(set(comments))  # optional: keep unique only

# Shuffle randomly
random.shuffle(comments)

# Save to output file
with open(output_file, "w", encoding="utf-8") as f:
    for c in comments:
        f.write(c + "\n")

print(f"✅ Merged {file1} and {file2} into {output_file}")
print(f"Total comments: {len(comments)}")


✅ Merged youtube_comments_highlightsNepal_romanized.txt and youtube_comments_highlightsNepal2_romanized.txt into merged_comments_highlightsNepal.txt
Total comments: 81035
